# DDPM, DDIM, and latent diffusion: what changes during sampling?

Training learns a denoising prediction. Sampling is the numerical procedure that turns that prediction into a clean sample. This lesson derives the reverse coefficients, trains a small denoiser on a two-mode distribution, compares stochastic DDPM sampling with deterministic DDIM sampling, and then places latent diffusion and classifier-free guidance in the same picture.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn
sns.set_theme(style='whitegrid',context='notebook'); torch.manual_seed(10); np.random.seed(10)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); T=60
beta=torch.linspace(1e-4,.02,T,device=device); alpha=1-beta; abar=torch.cumprod(alpha,0); abar_prev=torch.cat([torch.ones(1,device=device),abar[:-1]])
posterior_var=beta*(1-abar_prev)/(1-abar)
posterior_mean_x0=beta*torch.sqrt(abar_prev)/(1-abar); posterior_mean_xt=torch.sqrt(alpha)*(1-abar_prev)/(1-abar)

## 1. Why the DDPM posterior has these coefficients

The forward chain is `q(x_t|x_{t−1}) = N(√α_t x_{t−1}, β_t I)`. Because the chain is Gaussian, conditioning on both `x_t` and `x₀` gives another Gaussian:

`q(x_{t−1}|x_t,x₀) = N(μ_t(x_t,x₀), β̃_t I)`

with

`μ_t = [√ᾱ_{t−1} β_t/(1−ᾱ_t)]x₀ + [√α_t(1−ᾱ_{t−1})/(1−ᾱ_t)]x_t` and
`β̃_t = β_t(1−ᾱ_{t−1})/(1−ᾱ_t)`.

At sampling time `x₀` is unknown, so the model predicts noise and we turn that prediction into an estimate `x̂₀`.

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self,w=64): super().__init__(); self.w=w
    def forward(self,t):
        half=self.w//2; f=torch.exp(-torch.arange(half,device=t.device)*np.log(10000)/(half-1)); a=t.float().unsqueeze(1)*f.unsqueeze(0); return torch.cat([a.sin(),a.cos()],1)
class Denoiser(nn.Module):
    def __init__(self,w=64):
        super().__init__(); self.t=TimeEmbedding(w); self.m=nn.Sequential(nn.Linear(2+w,w),nn.SiLU(),nn.Linear(w,w),nn.SiLU(),nn.Linear(w,1))
    def forward(self,x,t): return self.m(torch.cat([x.unsqueeze(1),self.t(t/(T-1))],1)).squeeze(1)
model=Denoiser().to(device); opt=torch.optim.Adam(model.parameters(),lr=2e-3)
def sample_clean(n): return torch.where(torch.rand(n,device=device)>.5,torch.randn(n,device=device)*.18+1,torch.randn(n,device=device)*.18-1)
for step in range(1600):
    t=torch.randint(0,T,(256,),device=device); x0=sample_clean(256); eps=torch.randn_like(x0)
    xt=abar[t].sqrt()*x0+(1-abar[t]).sqrt()*eps
    loss=nn.functional.mse_loss(model(xt,t),eps); opt.zero_grad(); loss.backward(); opt.step()
print('training epsilon MSE:',float(loss))

## 2. DDPM sampling

At each reverse step, predict `ε`, recover `x̂₀`, compute the posterior mean, and add Gaussian noise with variance `β̃_t`. The noise is omitted at `t=0`; otherwise the chain would never settle.

In [ ]:
@torch.no_grad()
def predict_x0(xt,t): return (xt-(1-abar[t]).sqrt()*model(xt,t))/abar[t].sqrt()
@torch.no_grad()
def ddpm(n=1000):
    x=torch.randn(n,device=device); history=[x.cpu().numpy()]
    for t in range(T-1,-1,-1):
        x0hat=predict_x0(x,t); mean=posterior_mean_x0[t]*x0hat+posterior_mean_xt[t]*x
        if t>0: x=mean+posterior_var[t].sqrt()*torch.randn_like(x)
        else: x=mean
        history.append(x.cpu().numpy())
    return np.array(history)
trajectory=ddpm(); final=trajectory[-1]
print('sample mean/std:',round(final.mean(),3),round(final.std(),3)); plt.hist(final,bins=50,density=True); plt.title('DDPM samples'); plt.axvline(-1,color='k',ls=':'); plt.axvline(1,color='k',ls=':'); plt.show()

## 3. DDIM sampling and the `η` knob

DDIM keeps the same predicted clean estimate but chooses a different noise scale. In the common parameterization:

`x_{t−1} = √ᾱ_{t−1} x̂₀ + √(1−ᾱ_{t−1}−σ_t²) ε_θ(x_t,t) + σ_t z`.

`σ_t` is proportional to `η`. `η=1` gives a stochastic path similar in spirit to DDPM; `η=0` is deterministic for a fixed initial noise. DDIM can also skip time steps, trading computation against approximation error.

In [ ]:
@torch.no_grad()
def ddim(n=1000,eta=0.0,stride=1):
    x=torch.randn(n,device=device); history=[x.cpu().numpy()]; times=list(range(T-1,-1,-stride))
    if times[-1]!=0: times.append(0)
    for index,t in enumerate(times):
        x0hat=predict_x0(x,t); prev=0 if index==len(times)-1 else times[index+1]
        a_prev=abar[prev]; sigma=eta*torch.sqrt((1-a_prev)/(1-abar[t])*(1-abar[t]/a_prev)) if t>0 else torch.tensor(0.,device=device)
        direction=torch.sqrt(torch.clamp(1-a_prev-sigma**2,min=0))*model(x,torch.full_like(x,t))
        x=torch.sqrt(a_prev)*x0hat+direction
        if t>0 and eta>0: x=x+sigma*torch.randn_like(x)
        history.append(x.cpu().numpy())
    return np.array(history)

deterministic=ddim(1000,eta=0.0,stride=2)[-1]; stochastic=ddim(1000,eta=1.0,stride=2)[-1]
fig,axes=plt.subplots(1,2,figsize=(11,3),sharey=True)
axes[0].hist(deterministic,bins=50,density=True); axes[0].set_title('DDIM, η=0, 30 steps')
axes[1].hist(stochastic,bins=50,density=True); axes[1].set_title('DDIM noise, η=1, 30 steps')
for ax in axes: ax.axvline(-1,color='k',ls=':'); ax.axvline(1,color='k',ls=':')
plt.tight_layout()

## 4. Latent diffusion and classifier-free guidance

Latent diffusion inserts an autoencoder around the denoiser:

`image → encoder → z₀ → diffusion U-Net → ẑ₀ → decoder → image`.

The denoiser works on a smaller spatial tensor, reducing cost. Text conditioning enters through cross-attention: image-latent queries read text keys and values. Classifier-free guidance combines an unconditional and conditional prediction:

`ε_guided = ε_uncond + s(ε_cond − ε_uncond)`.

The scale `s` increases prompt adherence but can reduce diversity or create artifacts. It is a sampling choice, so it must be reported alongside the checkpoint, schedule, and number of steps.

In [ ]:
# The guidance operation is simple; its effect is a vector interpolation.
unconditional=torch.tensor([.2,.5,-.1]); conditional=torch.tensor([.8,.2,.4])
for scale in [0,1,3,7]: print(f's={scale}:',unconditional+scale*(conditional-unconditional))

### What this experiment establishes

DDPM and DDIM use the same denoising prediction but different reverse trajectories. DDPM injects posterior noise; DDIM can be deterministic and can skip steps. Latent diffusion changes the representation being denoised, while classifier-free guidance changes how conditional and unconditional predictions are combined.

### What it does not establish

A one-dimensional mixture cannot predict image quality. A real comparison needs fixed checkpoints, a held-out evaluation protocol, multiple seeds, compute reporting, and visual inspection of failure cases.